# Chapter 2 — Thermodynamic Foundations: Scientific Figures

**Figures:**
1. Gibbs energy surface and phase equilibrium
2. P-T phase diagram for a pure component
3. Fugacity coefficient vs pressure
4. Helmholtz energy decomposition

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({
    "font.family": "serif", "font.size": 9, "axes.labelsize": 10,
    "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8,
    "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in",
    "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3,
    "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150,
})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")

## Figure 2.1 — Gibbs Energy of Mixing and Phase Equilibrium

In [3]:
# Gibbs energy of mixing: ideal + excess (Margules model for illustration)
x = np.linspace(0.001, 0.999, 500)
R_val, T_val = 8.314, 300.0

# Ideal mixing
G_id = R_val * T_val * (x * np.log(x) + (1 - x) * np.log(1 - x))

# Excess: Margules A = 3000 J/mol (partially miscible)
A_marg = 7000.0
G_ex = A_marg * x * (1 - x)
G_mix = G_id + G_ex

# Common tangent (approximate)
x1_eq, x2_eq = 0.11, 0.89  # approximate equilibrium compositions
G1 = np.interp(x1_eq, x, G_mix)
G2 = np.interp(x2_eq, x, G_mix)
slope = (G2 - G1) / (x2_eq - x1_eq)
tangent = G1 + slope * (x - x1_eq)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 3.0))

ax1.plot(x, G_mix / 1000, color=BLUE, linewidth=1.5, label=r"$\Delta G_{\rm mix}$")
ax1.plot(x, G_id / 1000, color="grey", linewidth=0.8, linestyle="--", label=r"$\Delta G_{\rm id}$")
ax1.plot(x, tangent / 1000, color=ORANGE, linewidth=0.8, linestyle=":", label="Common tangent")
ax1.plot([x1_eq, x2_eq], [G1/1000, G2/1000], 'ko', markersize=5)
ax1.set_xlabel(r"Mole fraction $x_1$")
ax1.set_ylabel(r"$\Delta G_{\rm mix}$ (kJ/mol)")
ax1.set_title("(a) Gibbs energy of mixing")
ax1.legend(frameon=True, fontsize=7)
ax1.grid(True, alpha=0.3)

# Phase diagram: UCST system
T_range = np.linspace(260, 360, 100)
x1_curve = []
x2_curve = []
for T_i in T_range:
    # Margules: phase split when A > 2RT
    ratio = A_marg / (R_val * T_i)
    if ratio > 2:
        # Approximate spinodal from dG/dx = 0
        disc = max(0, 0.25 - 1.0 / (2.0 * ratio))
        x1_curve.append(0.5 - np.sqrt(disc))
        x2_curve.append(0.5 + np.sqrt(disc))
    else:
        x1_curve.append(0.5)
        x2_curve.append(0.5)

ax2.plot(x1_curve, T_range - 273.15, color=BLUE, linewidth=1.5)
ax2.plot(x2_curve, T_range - 273.15, color=BLUE, linewidth=1.5)
ax2.fill_betweenx(T_range - 273.15, x1_curve, x2_curve, alpha=0.1, color=BLUE)
ax2.set_xlabel(r"Mole fraction $x_1$")
ax2.set_ylabel("Temperature (\u00b0C)")
ax2.set_title("(b) Liquid\u2013liquid phase diagram")
ax2.annotate("Two\nphases", xy=(0.5, 20), ha="center", fontsize=8, color=BLUE)
ax2.annotate("Single\nphase", xy=(0.5, 75), ha="center", fontsize=8, color="grey")
ax2.grid(True, alpha=0.3)

fig.tight_layout()
save(fig, "fig_ch02_01_gibbs_mixing_phase_diagram.png")

Saved: fig_ch02_01_gibbs_mixing_phase_diagram.png


## Figure 2.2 — Fugacity Coefficient from NeqSim

Shows how the fugacity coefficient of water deviates from unity as
pressure increases, comparing CPA vs SRK.

In [4]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

pressures = np.arange(1, 305, 10)
phi_cpa = []
phi_srk = []

for P in pressures:
    try:
        f = SystemSrkCPAstatoil(273.15 + 150.0, float(P))
        f.addComponent("water", 1.0)
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        phi_cpa.append(float(f.getPhase(0).getComponent(0).getFugacityCoefficient()))
    except Exception:
        phi_cpa.append(np.nan)
    try:
        f2 = SystemSrkEos(273.15 + 150.0, float(P))
        f2.addComponent("water", 1.0)
        f2.setMixingRule("classic")
        ops2 = ThermodynamicOperations(f2)
        ops2.TPflash()
        f2.initProperties()
        phi_srk.append(float(f2.getPhase(0).getComponent(0).getFugacityCoefficient()))
    except Exception:
        phi_srk.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures, phi_cpa, color=BLUE, linewidth=1.5, label="CPA")
ax.plot(pressures, phi_srk, color=ORANGE, linewidth=1.5, linestyle="--", label="SRK")
ax.axhline(y=1.0, color="grey", linestyle=":", alpha=0.5)
ax.set_xlabel("Pressure (bar)")
ax.set_ylabel(r"Fugacity coefficient $\hat{\varphi}$")
ax.set_title(r"Water $\hat{\varphi}$ at 150\u00b0C")
ax.legend(frameon=True); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch02_02_fugacity_coefficient.png")

Saved: fig_ch02_02_fugacity_coefficient.png


## Figure 2.3 — Helmholtz Energy Decomposition

Conceptual diagram showing the additive contributions to the Helmholtz
energy in the CPA framework.

In [5]:
fig, ax = plt.subplots(figsize=(5.0, 2.5))

contributions = [
    (r"$A^{\rm ig}$", "Ideal gas", 0.7, "#d9d9d9"),
    (r"$A^{\rm cubic}$", "Cubic (SRK)", 0.5, BLUE),
    (r"$A^{\rm assoc}$", "Association", 0.3, ORANGE),
]

bars = ax.barh([c[1] for c in contributions],
               [-c[2] for c in contributions],
               color=[c[3] for c in contributions],
               edgecolor="white", linewidth=0.5, height=0.5)

for bar, (label, desc, val, col) in zip(bars, contributions):
    ax.text(bar.get_width() - 0.02, bar.get_y() + bar.get_height()/2,
            label, ha="right", va="center", fontsize=10, fontweight="bold",
            color="white" if col != "#d9d9d9" else "black")

ax.set_xlabel(r"$A / nRT$ (dimensionless)")
ax.set_title("CPA Helmholtz Energy Decomposition")
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_xlim(-1.0, 0.1)

# Total
total = -sum(c[2] for c in contributions)
ax.annotate(f"Total: {total:.1f}", xy=(total, -0.5), fontsize=9,
            fontweight="bold", ha="center")

fig.tight_layout()
save(fig, "fig_ch02_03_helmholtz_decomposition.png")

Saved: fig_ch02_03_helmholtz_decomposition.png
